# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

## Data

In [2]:
data_path = '..\\..\\..\\data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [3]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        left_on=['Season', 'Team1', 'Team2'],
        right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
    tourney_data['T1_TeamID'] = tourney_data['Team1']
    tourney_data['T2_TeamID'] = tourney_data['Team2']
    tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage1,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage1, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            elo = pd.read_csv(join(data_path, 'elo.csv'))
            elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
            elo = elo.drop(['CoachName'], axis=1)
            def add_elo_column(df):
                df = df.copy()
                df = pd.merge(
                        df,
                        elo[['Season', 'DayNum', 'TeamID', 'TeamELO', 'CoachELO']],
                        left_on=['Season', 'DayNum', 'T1_TeamID'],
                        right_on=['Season', 'DayNum', 'TeamID'],
                        how='left'
                    )
                df = df.drop(['TeamID'], axis=1)
                return df
            df_train = add_elo_column(df_train)
            df_test = add_elo_column(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

## Get data to use models on

In [23]:
columns_to_include_women = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
#  'CoachELO'
]
columns_to_include_men = [
 'Season',
 'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'TeamELO',
 'CoachELO'
]

year_range = [2023]
start_season = 2003 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

        x_train_women, y_train_women = clear_na_from_x_y(x_train_women.copy(), y_train_women.copy())
        x_test_women, y_test_women = clear_na_from_x_y(x_test_women.copy(), y_test_women.copy())
        x_train_men, y_train_men = clear_na_from_x_y(x_train_men.copy(), y_train_men.copy())
        x_test_men, y_test_men = clear_na_from_x_y(x_test_men.copy(), y_test_men.copy())

## Which models to consider

In [24]:
def train_and_evaluate(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_test)[:, 1]  # Get probability of class 1
    score = brier_score_loss(y_test, y_pred)
    return score

# Initialize models
catboost_model = CatBoostClassifier(verbose=0, iterations=500, depth=6, learning_rate=0.05)
xgboost_model = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, use_label_encoder=False, eval_metric='logloss')
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, alpha=0.01)
lr_model = LogisticRegression()

# Train and evaluate for women
y_pred_women_catboost = train_and_evaluate(catboost_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_xgboost = train_and_evaluate(xgboost_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_mlp = train_and_evaluate(mlp_model, x_train_women, y_train_women, x_test_women, y_test_women)
y_pred_women_lr = train_and_evaluate(lr_model, x_train_women, y_train_women, x_test_women, y_test_women)

# Train and evaluate for men
y_pred_men_catboost = train_and_evaluate(catboost_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_xgboost = train_and_evaluate(xgboost_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_mlp = train_and_evaluate(mlp_model, x_train_men, y_train_men, x_test_men, y_test_men)
y_pred_men_lr = train_and_evaluate(lr_model, x_train_men, y_train_men, x_test_men, y_test_men)

print("Brier Scores:")
print(f"CatBoost (Women): {y_pred_women_catboost}")
print(f"XGBoost (Women): {y_pred_women_xgboost}")
print(f"MLP (Women): {y_pred_women_mlp}")
print(f"Logistic (Women): {y_pred_women_lr}")
print(f"CatBoost (Men): {y_pred_men_catboost}")
print(f"XGBoost (Men): {y_pred_men_xgboost}")
print(f"MLP (Men): {y_pred_men_mlp}")
print(f"Logistic (Men): {y_pred_men_lr}")

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Brier Scores:
CatBoost (Women): 0.19789686715342686
XGBoost (Women): 0.21424128811938234
MLP (Women): 0.2537342770284451
Logistic (Women): 0.1612005025608674
CatBoost (Men): 0.22776204616428086
XGBoost (Men): 0.23156317233882792
MLP (Men): 0.41228070127153676
Logistic (Men): 0.20392680092784002


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Training models

## CatBoost

In [25]:
# FOR WOMEN
def objective(trial, x_train, y_train, x_val, y_val):
    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "verbose": 0,
        "loss_function": "Logloss"
    }
    model = CatBoostClassifier(**params)
    model.fit(x_train, y_train, eval_set=(x_val, y_val), early_stopping_rounds=50, verbose=False)
    
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

###################################################### WOMEN

# Split data into training and validation for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(x_train_women, y_train_women, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Train final model with best hyperparameters
best_params_women = study.best_params
best_catboost_women = CatBoostClassifier(**best_params_women)
best_catboost_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women = best_catboost_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women = brier_score_loss(y_test_women, y_pred_women)
print("Best CatBoost Brier Score (Women):", brier_women)

###################################################### MEN

# Split data into training and validation for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(x_train_men, y_train_men, test_size=0.2, random_state=42)

# Run optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Train final model with best hyperparameters
best_params_men = study.best_params
best_catboost_men = CatBoostClassifier(**best_params_men)
best_catboost_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men = best_catboost_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men = brier_score_loss(y_test_men, y_pred_men)
print("Best CatBoost Brier Score (men):", brier_men)

[I 2025-03-19 20:06:54,703] A new study created in memory with name: no-name-16028b53-0204-457a-bd0f-e265f7e29878
[I 2025-03-19 20:06:55,102] Trial 0 finished with value: 0.14136633400166013 and parameters: {'iterations': 2778, 'learning_rate': 0.08793228037269027, 'depth': 6, 'l2_leaf_reg': 0.02970598787968118, 'border_count': 176, 'random_strength': 0.0021167027206135005, 'bagging_temperature': 0.6281753251185674}. Best is trial 0 with value: 0.14136633400166013.
[I 2025-03-19 20:06:55,754] Trial 1 finished with value: 0.15497982693365345 and parameters: {'iterations': 1082, 'learning_rate': 0.19010785480575323, 'depth': 7, 'l2_leaf_reg': 0.03499099628381493, 'border_count': 149, 'random_strength': 7.437229267774547, 'bagging_temperature': 0.23715984146546}. Best is trial 0 with value: 0.14136633400166013.
[I 2025-03-19 20:06:56,951] Trial 2 finished with value: 0.14205338378953572 and parameters: {'iterations': 2553, 'learning_rate': 0.051691008269728146, 'depth': 7, 'l2_leaf_reg': 

[I 2025-03-19 20:08:04,331] Trial 23 finished with value: 0.1368230509218631 and parameters: {'iterations': 1544, 'learning_rate': 0.02352438090898458, 'depth': 4, 'l2_leaf_reg': 0.21898451529892562, 'border_count': 190, 'random_strength': 0.9236326063394052, 'bagging_temperature': 0.48079419560206205}. Best is trial 14 with value: 0.13559244076860102.
[I 2025-03-19 20:08:05,947] Trial 24 finished with value: 0.13749992375493042 and parameters: {'iterations': 1138, 'learning_rate': 0.030171787185970377, 'depth': 4, 'l2_leaf_reg': 1.0754077750134097, 'border_count': 188, 'random_strength': 4.145053828589941, 'bagging_temperature': 0.4825019418307252}. Best is trial 14 with value: 0.13559244076860102.
[I 2025-03-19 20:08:07,380] Trial 25 finished with value: 0.13782827122801655 and parameters: {'iterations': 1953, 'learning_rate': 0.02339467012581293, 'depth': 4, 'l2_leaf_reg': 0.3931160433133884, 'border_count': 251, 'random_strength': 0.39784720968183845, 'bagging_temperature': 0.60822

[I 2025-03-19 20:08:54,503] Trial 47 finished with value: 0.14229461440418137 and parameters: {'iterations': 1077, 'learning_rate': 0.16092519046332487, 'depth': 6, 'l2_leaf_reg': 0.007681992872485718, 'border_count': 213, 'random_strength': 1.525925615949145, 'bagging_temperature': 0.6236912723504733}. Best is trial 33 with value: 0.13347083100588572.
[I 2025-03-19 20:08:55,752] Trial 48 finished with value: 0.13812998286666325 and parameters: {'iterations': 1781, 'learning_rate': 0.03454185485648789, 'depth': 4, 'l2_leaf_reg': 0.2838029580567791, 'border_count': 198, 'random_strength': 3.4828700381693083, 'bagging_temperature': 0.7782851352624843}. Best is trial 33 with value: 0.13347083100588572.
[I 2025-03-19 20:08:56,486] Trial 49 finished with value: 0.13705535744996775 and parameters: {'iterations': 1487, 'learning_rate': 0.059804899117179594, 'depth': 5, 'l2_leaf_reg': 3.3922067255165675, 'border_count': 228, 'random_strength': 0.9503506271366726, 'bagging_temperature': 0.53331

0:	learn: 0.6797275	total: 11.6ms	remaining: 11.3s
1:	learn: 0.6633603	total: 22.1ms	remaining: 10.8s
2:	learn: 0.6523133	total: 32.2ms	remaining: 10.4s
3:	learn: 0.6426953	total: 42.4ms	remaining: 10.3s
4:	learn: 0.6305053	total: 52.8ms	remaining: 10.2s
5:	learn: 0.6216808	total: 62.7ms	remaining: 10.1s
6:	learn: 0.6143751	total: 74.4ms	remaining: 10.3s
7:	learn: 0.6050030	total: 85.8ms	remaining: 10.4s
8:	learn: 0.5965678	total: 96.9ms	remaining: 10.4s
9:	learn: 0.5883286	total: 108ms	remaining: 10.4s
10:	learn: 0.5796636	total: 118ms	remaining: 10.3s
11:	learn: 0.5705924	total: 128ms	remaining: 10.3s
12:	learn: 0.5633609	total: 138ms	remaining: 10.2s
13:	learn: 0.5557115	total: 148ms	remaining: 10.2s
14:	learn: 0.5472390	total: 159ms	remaining: 10.2s
15:	learn: 0.5409535	total: 169ms	remaining: 10.1s
16:	learn: 0.5356137	total: 179ms	remaining: 10.1s
17:	learn: 0.5288461	total: 191ms	remaining: 10.1s
18:	learn: 0.5212789	total: 202ms	remaining: 10.2s
19:	learn: 0.5146903	total: 213m

178:	learn: 0.2275877	total: 1.97s	remaining: 8.75s
179:	learn: 0.2261235	total: 1.98s	remaining: 8.73s
180:	learn: 0.2248394	total: 1.99s	remaining: 8.72s
181:	learn: 0.2240710	total: 2s	remaining: 8.7s
182:	learn: 0.2235851	total: 2.01s	remaining: 8.69s
183:	learn: 0.2228401	total: 2.02s	remaining: 8.67s
184:	learn: 0.2223316	total: 2.03s	remaining: 8.66s
185:	learn: 0.2215731	total: 2.04s	remaining: 8.64s
186:	learn: 0.2210321	total: 2.05s	remaining: 8.63s
187:	learn: 0.2200873	total: 2.06s	remaining: 8.63s
188:	learn: 0.2184729	total: 2.07s	remaining: 8.62s
189:	learn: 0.2172901	total: 2.08s	remaining: 8.6s
190:	learn: 0.2165658	total: 2.09s	remaining: 8.59s
191:	learn: 0.2158822	total: 2.1s	remaining: 8.58s
192:	learn: 0.2150414	total: 2.11s	remaining: 8.56s
193:	learn: 0.2141182	total: 2.12s	remaining: 8.55s
194:	learn: 0.2132284	total: 2.13s	remaining: 8.53s
195:	learn: 0.2120246	total: 2.14s	remaining: 8.52s
196:	learn: 0.2112228	total: 2.15s	remaining: 8.51s
197:	learn: 0.2108

...

2014:	learn: 0.1189977	total: 10.7s	remaining: 4.15s
2015:	learn: 0.1189437	total: 10.7s	remaining: 4.15s
2016:	learn: 0.1187997	total: 10.7s	remaining: 4.14s
2017:	learn: 0.1187279	total: 10.7s	remaining: 4.13s
2018:	learn: 0.1186276	total: 10.7s	remaining: 4.13s
2019:	learn: 0.1184977	total: 10.7s	remaining: 4.12s
2020:	learn: 0.1183873	total: 10.7s	remaining: 4.12s
2021:	learn: 0.1182878	total: 10.7s	remaining: 4.11s
2022:	learn: 0.1182052	total: 10.7s	remaining: 4.11s
2023:	learn: 0.1181250	total: 10.8s	remaining: 4.1s
2024:	learn: 0.1180430	total: 10.8s	remaining: 4.09s
2025:	learn: 0.1179209	total: 10.8s	remaining: 4.09s
2026:	learn: 0.1178466	total: 10.8s	remaining: 4.08s
2027:	learn: 0.1177850	total: 10.8s	remaining: 4.08s
2028:	learn: 0.1177182	total: 10.8s	remaining: 4.07s
2029:	learn: 0.1176620	total: 10.8s	remaining: 4.07s
2030:	learn: 0.1175045	total: 10.8s	remaining: 4.06s
2031:	learn: 0.1174384	total: 10.8s	remaining: 4.06s
2032:	learn: 0.1173275	total: 10.8s	remaining: 

2181:	learn: 0.1050741	total: 11.6s	remaining: 3.26s
2182:	learn: 0.1049795	total: 11.6s	remaining: 3.26s
2183:	learn: 0.1048985	total: 11.6s	remaining: 3.25s
2184:	learn: 0.1047883	total: 11.6s	remaining: 3.25s
2185:	learn: 0.1047032	total: 11.6s	remaining: 3.24s
2186:	learn: 0.1046371	total: 11.6s	remaining: 3.23s
2187:	learn: 0.1045652	total: 11.6s	remaining: 3.23s
2188:	learn: 0.1044757	total: 11.6s	remaining: 3.22s
2189:	learn: 0.1043788	total: 11.6s	remaining: 3.22s
2190:	learn: 0.1042972	total: 11.6s	remaining: 3.21s
2191:	learn: 0.1042212	total: 11.6s	remaining: 3.21s
2192:	learn: 0.1041539	total: 11.7s	remaining: 3.2s
2193:	learn: 0.1041009	total: 11.7s	remaining: 3.2s
2194:	learn: 0.1040206	total: 11.7s	remaining: 3.19s
2195:	learn: 0.1039132	total: 11.7s	remaining: 3.19s
2196:	learn: 0.1038171	total: 11.7s	remaining: 3.18s
2197:	learn: 0.1037235	total: 11.7s	remaining: 3.18s
2198:	learn: 0.1036758	total: 11.7s	remaining: 3.17s
2199:	learn: 0.1035976	total: 11.7s	remaining: 3

2352:	learn: 0.0924902	total: 12.5s	remaining: 2.35s
2353:	learn: 0.0924258	total: 12.5s	remaining: 2.35s
2354:	learn: 0.0923792	total: 12.5s	remaining: 2.34s
2355:	learn: 0.0922818	total: 12.5s	remaining: 2.33s
2356:	learn: 0.0922067	total: 12.5s	remaining: 2.33s
2357:	learn: 0.0921246	total: 12.5s	remaining: 2.32s
2358:	learn: 0.0920574	total: 12.5s	remaining: 2.32s
2359:	learn: 0.0920112	total: 12.5s	remaining: 2.31s
2360:	learn: 0.0919368	total: 12.5s	remaining: 2.31s
2361:	learn: 0.0918972	total: 12.5s	remaining: 2.3s
2362:	learn: 0.0918515	total: 12.5s	remaining: 2.3s
2363:	learn: 0.0917764	total: 12.6s	remaining: 2.29s
2364:	learn: 0.0916933	total: 12.6s	remaining: 2.29s
2365:	learn: 0.0916391	total: 12.6s	remaining: 2.28s
2366:	learn: 0.0915580	total: 12.6s	remaining: 2.28s
2367:	learn: 0.0914907	total: 12.6s	remaining: 2.27s
2368:	learn: 0.0914121	total: 12.6s	remaining: 2.27s
2369:	learn: 0.0913498	total: 12.6s	remaining: 2.26s
2370:	learn: 0.0913024	total: 12.6s	remaining: 2

2511:	learn: 0.0823698	total: 13.4s	remaining: 1.51s
2512:	learn: 0.0822966	total: 13.4s	remaining: 1.51s
2513:	learn: 0.0822451	total: 13.4s	remaining: 1.5s
2514:	learn: 0.0821907	total: 13.4s	remaining: 1.5s
2515:	learn: 0.0821255	total: 13.4s	remaining: 1.49s
2516:	learn: 0.0820499	total: 13.4s	remaining: 1.49s
2517:	learn: 0.0820015	total: 13.4s	remaining: 1.48s
2518:	learn: 0.0819318	total: 13.5s	remaining: 1.48s
2519:	learn: 0.0818674	total: 13.5s	remaining: 1.47s
2520:	learn: 0.0818085	total: 13.5s	remaining: 1.47s
2521:	learn: 0.0817568	total: 13.5s	remaining: 1.46s
2522:	learn: 0.0816796	total: 13.5s	remaining: 1.46s
2523:	learn: 0.0816356	total: 13.5s	remaining: 1.45s
2524:	learn: 0.0815902	total: 13.5s	remaining: 1.45s
2525:	learn: 0.0815184	total: 13.5s	remaining: 1.44s
2526:	learn: 0.0814670	total: 13.5s	remaining: 1.44s
2527:	learn: 0.0814111	total: 13.5s	remaining: 1.43s
2528:	learn: 0.0813637	total: 13.5s	remaining: 1.43s
2529:	learn: 0.0812794	total: 13.5s	remaining: 1

2695:	learn: 0.0722708	total: 14.5s	remaining: 537ms
2696:	learn: 0.0722229	total: 14.5s	remaining: 532ms
2697:	learn: 0.0721801	total: 14.5s	remaining: 527ms
2698:	learn: 0.0721235	total: 14.5s	remaining: 521ms
2699:	learn: 0.0720929	total: 14.5s	remaining: 516ms
2700:	learn: 0.0720436	total: 14.5s	remaining: 510ms
2701:	learn: 0.0720037	total: 14.5s	remaining: 505ms
2702:	learn: 0.0719363	total: 14.5s	remaining: 500ms
2703:	learn: 0.0718845	total: 14.5s	remaining: 494ms
2704:	learn: 0.0718211	total: 14.5s	remaining: 489ms
2705:	learn: 0.0717735	total: 14.5s	remaining: 483ms
2706:	learn: 0.0717259	total: 14.5s	remaining: 478ms
2707:	learn: 0.0716841	total: 14.5s	remaining: 473ms
2708:	learn: 0.0716448	total: 14.6s	remaining: 467ms
2709:	learn: 0.0716098	total: 14.6s	remaining: 462ms
2710:	learn: 0.0715580	total: 14.6s	remaining: 457ms
2711:	learn: 0.0714998	total: 14.6s	remaining: 451ms
2712:	learn: 0.0714507	total: 14.6s	remaining: 446ms
2713:	learn: 0.0713984	total: 14.6s	remaining:

## XGBoost

In [27]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split

############################################
# FOR WOMEN
############################################
def objective_xgb(trial, x_train, y_train, x_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "verbosity": 0
    }
    
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", early_stopping_rounds=50)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split women's training data for hyperparameter tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run optimization for women
study_women = optuna.create_study(direction="minimize")
study_women.optimize(lambda trial: objective_xgb(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)
best_params_women_xgb = study_women.best_params
print("Best XGBoost Params (Women):", best_params_women_xgb)

# Train final XGBoost model for women with best hyperparameters
best_xgb_women = XGBClassifier(**best_params_women_xgb, use_label_encoder=False, eval_metric="logloss")
best_xgb_women.fit(x_train_women, y_train_women)
y_pred_women_xgb = best_xgb_women.predict_proba(x_test_women)[:, 1]
brier_women_xgb = brier_score_loss(y_test_women, y_pred_women_xgb)
print("Best XGBoost Brier Score (Women):", brier_women_xgb)

############################################
# FOR MEN
############################################
def objective_xgb_men(trial, x_train, y_train, x_val, y_val):
    # We can use the same search space as for women
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 20),
        "verbosity": 0
    }
    
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", early_stopping_rounds=50)
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split men's training data for hyperparameter tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run optimization for men
study_men = optuna.create_study(direction="minimize")
study_men.optimize(lambda trial: objective_xgb_men(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)
best_params_men_xgb = study_men.best_params
print("Best XGBoost Params (Men):", best_params_men_xgb)

# Train final XGBoost model for men with best hyperparameters
best_xgb_men = XGBClassifier(**best_params_men_xgb, use_label_encoder=False, eval_metric="logloss")
best_xgb_men.fit(x_train_men, y_train_men)
y_pred_men_xgb = best_xgb_men.predict_proba(x_test_men)[:, 1]
brier_men_xgb = brier_score_loss(y_test_men, y_pred_men_xgb)
print("Best XGBoost Brier Score (Men):", brier_men_xgb)


[I 2025-03-19 20:15:53,034] A new study created in memory with name: no-name-01de20d5-ee7a-4d9e-b081-1fcda8c61286
[I 2025-03-19 20:15:54,089] Trial 0 finished with value: 0.13461643573936943 and parameters: {'n_estimators': 764, 'max_depth': 9, 'learning_rate': 0.020231144230404813, 'subsample': 0.7200403788432235, 'colsample_bytree': 0.779543595622037, 'min_child_weight': 7}. Best is trial 0 with value: 0.13461643573936943.
[I 2025-03-19 20:15:54,711] Trial 1 finished with value: 0.13587683954944815 and parameters: {'n_estimators': 794, 'max_depth': 5, 'learning_rate': 0.03336749112907669, 'subsample': 0.8612278113603571, 'colsample_bytree': 0.6174658874488385, 'min_child_weight': 2}. Best is trial 0 with value: 0.13461643573936943.
[I 2025-03-19 20:15:54,911] Trial 2 finished with value: 0.13513050678666802 and parameters: {'n_estimators': 168, 'max_depth': 3, 'learning_rate': 0.10992667987128452, 'subsample': 0.6266646003034386, 'colsample_bytree': 0.9013139924482795, 'min_child_wei

[I 2025-03-19 20:16:11,577] Trial 26 finished with value: 0.13228000748395108 and parameters: {'n_estimators': 643, 'max_depth': 9, 'learning_rate': 0.02736566727930523, 'subsample': 0.7980376943336807, 'colsample_bytree': 0.9714886571551237, 'min_child_weight': 8}. Best is trial 26 with value: 0.13228000748395108.
[I 2025-03-19 20:16:12,487] Trial 27 finished with value: 0.13399987844214478 and parameters: {'n_estimators': 629, 'max_depth': 9, 'learning_rate': 0.02662136402753905, 'subsample': 0.8035975871119937, 'colsample_bytree': 0.9538927536706223, 'min_child_weight': 8}. Best is trial 26 with value: 0.13228000748395108.
[I 2025-03-19 20:16:13,652] Trial 28 finished with value: 0.13387357928980972 and parameters: {'n_estimators': 516, 'max_depth': 10, 'learning_rate': 0.016104946705946994, 'subsample': 0.7638233771606368, 'colsample_bytree': 0.9786557218054663, 'min_child_weight': 9}. Best is trial 26 with value: 0.13228000748395108.
[I 2025-03-19 20:16:14,164] Trial 29 finished w

Best XGBoost Params (Women): {'n_estimators': 933, 'max_depth': 3, 'learning_rate': 0.22519085532254396, 'subsample': 0.8245232021162286, 'colsample_bytree': 0.8013580807879005, 'min_child_weight': 3}


[I 2025-03-19 20:16:27,283] A new study created in memory with name: no-name-a48053d6-e827-48af-b776-f471aaaa7521


Best XGBoost Brier Score (Women): 0.24446490692797904


[I 2025-03-19 20:16:27,741] Trial 0 finished with value: 0.18424760161554057 and parameters: {'n_estimators': 238, 'max_depth': 7, 'learning_rate': 0.1490760591869303, 'subsample': 0.7766305930125168, 'colsample_bytree': 0.9787934591160615, 'min_child_weight': 6}. Best is trial 0 with value: 0.18424760161554057.
[I 2025-03-19 20:16:28,923] Trial 1 finished with value: 0.18259703754394074 and parameters: {'n_estimators': 338, 'max_depth': 7, 'learning_rate': 0.03158898440962355, 'subsample': 0.947093441516559, 'colsample_bytree': 0.6085767710400742, 'min_child_weight': 1}. Best is trial 1 with value: 0.18259703754394074.
[I 2025-03-19 20:16:31,996] Trial 2 finished with value: 0.18125364098862376 and parameters: {'n_estimators': 807, 'max_depth': 7, 'learning_rate': 0.011112896699235991, 'subsample': 0.8837440387100757, 'colsample_bytree': 0.6537994319269786, 'min_child_weight': 1}. Best is trial 2 with value: 0.18125364098862376.
[I 2025-03-19 20:16:32,454] Trial 3 finished with value:

[I 2025-03-19 20:17:01,544] Trial 26 finished with value: 0.18038243197822382 and parameters: {'n_estimators': 580, 'max_depth': 8, 'learning_rate': 0.029401485331365777, 'subsample': 0.6885839921529144, 'colsample_bytree': 0.6875097185986856, 'min_child_weight': 7}. Best is trial 18 with value: 0.1762582039865362.
[I 2025-03-19 20:17:02,303] Trial 27 finished with value: 0.17959394650435967 and parameters: {'n_estimators': 447, 'max_depth': 6, 'learning_rate': 0.04471788441242614, 'subsample': 0.6458100062022121, 'colsample_bytree': 0.6183304779518072, 'min_child_weight': 9}. Best is trial 18 with value: 0.1762582039865362.
[I 2025-03-19 20:17:02,959] Trial 28 finished with value: 0.18431091553468126 and parameters: {'n_estimators': 597, 'max_depth': 10, 'learning_rate': 0.08422131915052813, 'subsample': 0.6817472214953906, 'colsample_bytree': 0.6736270999894943, 'min_child_weight': 8}. Best is trial 18 with value: 0.1762582039865362.
[I 2025-03-19 20:17:03,453] Trial 29 finished with

Best XGBoost Params (Men): {'n_estimators': 601, 'max_depth': 9, 'learning_rate': 0.0830576079307918, 'subsample': 0.601815152260508, 'colsample_bytree': 0.7297664514409565, 'min_child_weight': 8}
Best XGBoost Brier Score (Men): 0.24635877308537615


## Logistic regression

In [28]:
def objective_lr(trial, x_train, y_train, x_val, y_val):
    C = trial.suggest_float("C", 1e-4, 10, log=True)
    
    # Define the model pipeline with scaling
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=C, solver="liblinear", max_iter=1000))
    ])
    
    # Train the model
    model.fit(x_train, y_train)
    
    # Predict probabilities
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score as the optimization metric
    return brier_score_loss(y_val, y_pred)


###################################################### WOMEN

# Split data for tuning
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women), n_trials=50)

# Get best parameters
best_params_women_lr = study.best_params
print("Best Logistic Regression Params (Women):", best_params_women_lr)

# Train final Logistic Regression model with best params
best_lr_women = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_women_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_women.fit(x_train_women, y_train_women)

# Predict on test set
y_pred_women_lr = best_lr_women.predict_proba(x_test_women)[:, 1]

# Compute Brier Score
brier_women_lr = brier_score_loss(y_test_women, y_pred_women_lr)
print("Best Logistic Regression Brier Score (Women):", brier_women_lr)


###################################################### MEN

# Split data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

# Run Optuna optimization
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_lr(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men), n_trials=50)

# Get best parameters
best_params_men_lr = study.best_params
print("Best Logistic Regression Params (Men):", best_params_men_lr)

# Train final Logistic Regression model with best params
best_lr_men = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(C=best_params_men_lr["C"], solver="liblinear", max_iter=1000))
])
best_lr_men.fit(x_train_men, y_train_men)

# Predict on test set
y_pred_men_lr = best_lr_men.predict_proba(x_test_men)[:, 1]

# Compute Brier Score
brier_men_lr = brier_score_loss(y_test_men, y_pred_men_lr)
print("Best Logistic Regression Brier Score (Men):", brier_men_lr)

[I 2025-03-19 20:17:50,093] A new study created in memory with name: no-name-7eacc1de-54cd-4f24-87b4-6dc7e58eb299
[I 2025-03-19 20:17:50,105] Trial 0 finished with value: 0.13847840178357232 and parameters: {'C': 0.02239499444774401}. Best is trial 0 with value: 0.13847840178357232.
[I 2025-03-19 20:17:50,113] Trial 1 finished with value: 0.14001090961443394 and parameters: {'C': 0.011171170411041507}. Best is trial 0 with value: 0.13847840178357232.
[I 2025-03-19 20:17:50,125] Trial 2 finished with value: 0.1412824336033557 and parameters: {'C': 5.978878390514815}. Best is trial 0 with value: 0.13847840178357232.
[I 2025-03-19 20:17:50,143] Trial 3 finished with value: 0.2103966589148487 and parameters: {'C': 0.0002612035296682175}. Best is trial 0 with value: 0.13847840178357232.
[I 2025-03-19 20:17:50,160] Trial 4 finished with value: 0.13958658859822023 and parameters: {'C': 0.3785332728616949}. Best is trial 0 with value: 0.13847840178357232.
[I 2025-03-19 20:17:50,166] Trial 5 fi

[I 2025-03-19 20:17:50,975] Trial 48 finished with value: 0.1386752582012205 and parameters: {'C': 0.1212361554714396}. Best is trial 41 with value: 0.1381861945885923.
[I 2025-03-19 20:17:51,005] Trial 49 finished with value: 0.13935727581784815 and parameters: {'C': 0.28693107066686346}. Best is trial 41 with value: 0.1381861945885923.
[I 2025-03-19 20:17:51,021] A new study created in memory with name: no-name-9bae0b62-48f0-44e8-a326-c82bdac6e2b9
[I 2025-03-19 20:17:51,040] Trial 0 finished with value: 0.17946324843421232 and parameters: {'C': 0.0064495377753209255}. Best is trial 0 with value: 0.17946324843421232.
[I 2025-03-19 20:17:51,065] Trial 1 finished with value: 0.17774266995896454 and parameters: {'C': 0.09377895208801401}. Best is trial 1 with value: 0.17774266995896454.
[I 2025-03-19 20:17:51,089] Trial 2 finished with value: 0.17860177590471382 and parameters: {'C': 0.3623920384380762}. Best is trial 1 with value: 0.17774266995896454.
[I 2025-03-19 20:17:51,107] Trial 3

Best Logistic Regression Params (Women): {'C': 0.04025750295084394}
Best Logistic Regression Brier Score (Women): 0.1607962687934984


[I 2025-03-19 20:17:51,219] Trial 7 finished with value: 0.17871905214132552 and parameters: {'C': 0.00840945156540377}. Best is trial 1 with value: 0.17774266995896454.
[I 2025-03-19 20:17:51,236] Trial 8 finished with value: 0.2226604154161663 and parameters: {'C': 0.00019175580506399943}. Best is trial 1 with value: 0.17774266995896454.
[I 2025-03-19 20:17:51,252] Trial 9 finished with value: 0.1903189748843878 and parameters: {'C': 0.0013527541766047155}. Best is trial 1 with value: 0.17774266995896454.
[I 2025-03-19 20:17:51,286] Trial 10 finished with value: 0.17931786619077858 and parameters: {'C': 9.922445724584742}. Best is trial 1 with value: 0.17774266995896454.
[I 2025-03-19 20:17:51,319] Trial 11 finished with value: 0.17740538017326968 and parameters: {'C': 0.046672493767347105}. Best is trial 11 with value: 0.17740538017326968.
[I 2025-03-19 20:17:51,353] Trial 12 finished with value: 0.17742134811958793 and parameters: {'C': 0.023142746952895657}. Best is trial 11 with 

Best Logistic Regression Params (Men): {'C': 0.03289119677600514}
Best Logistic Regression Brier Score (Men): 0.2041976372634142


## Neural Net

In [29]:
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
import numpy as np

# ------------------------------ FOR WOMEN ------------------------------

def objective_nn(trial, x_train, y_train, x_val, y_val):
    # Define hyperparameters for the neural network.
    # Hidden layers: We choose the number of layers and neurons per layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_layer_sizes = []
    for i in range(n_layers):
        neurons = trial.suggest_int(f"n_units_l{i}", 10, 200)
        hidden_layer_sizes.append(neurons)
    hidden_layer_sizes = tuple(hidden_layer_sizes)
    
    # Regularization strength
    alpha = trial.suggest_float("alpha", 1e-2, 1.0, log=True)
    
    # Initial learning rate
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
    
    # Activation function: 'relu' or 'tanh' are common choices.
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    
    # Build a pipeline with StandardScaler and MLPClassifier
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000,
            random_state=42
        ))
    ])
    
    # Fit the model on the training split
    model.fit(x_train, y_train)
    
    # Predict probabilities on the validation split
    y_pred = model.predict_proba(x_val)[:, 1]
    
    # Return Brier Score (lower is better)
    return brier_score_loss(y_val, y_pred)

# Split women's training data into train and validation sets
x_train_women_tr, x_val_women, y_train_women_tr, y_val_women = train_test_split(
    x_train_women, y_train_women, test_size=0.2, random_state=42
)

# Optimize neural network hyperparameters for women
study_women_nn = optuna.create_study(direction="minimize")
study_women_nn.optimize(lambda trial: objective_nn(trial, x_train_women_tr, y_train_women_tr, x_val_women, y_val_women),
                          n_trials=50)
best_params_women_nn = study_women_nn.best_params
print("Best NN Params (Women):", best_params_women_nn)

# Train final NN model for women with best hyperparameters
best_nn_women = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=tuple(best_params_women_nn[f"n_units_l{i}"] for i in range(best_params_women_nn["n_layers"])),
        activation=best_params_women_nn["activation"],
        alpha=best_params_women_nn["alpha"],
        learning_rate_init=best_params_women_nn["learning_rate_init"],
        max_iter=1000,
        random_state=42
    ))
])
best_nn_women.fit(x_train_women, y_train_women)
y_pred_women_nn = best_nn_women.predict_proba(x_test_women)[:, 1]
brier_women_nn = brier_score_loss(y_test_women, y_pred_women_nn)
print("Best NN Brier Score (Women):", brier_women_nn)

# ------------------------------ FOR MEN ------------------------------

def objective_nn_men(trial, x_train, y_train, x_val, y_val):
    # Use the same search space as for women
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_layer_sizes = []
    for i in range(n_layers):
        neurons = trial.suggest_int(f"n_units_l{i}", 10, 100)
        hidden_layer_sizes.append(neurons)
    hidden_layer_sizes = tuple(hidden_layer_sizes)
    
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])
    
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=1000,
            random_state=42
        ))
    ])
    model.fit(x_train, y_train)
    y_pred = model.predict_proba(x_val)[:, 1]
    return brier_score_loss(y_val, y_pred)

# Split men's training data for tuning
x_train_men_tr, x_val_men, y_train_men_tr, y_val_men = train_test_split(
    x_train_men, y_train_men, test_size=0.2, random_state=42
)

study_men_nn = optuna.create_study(direction="minimize")
study_men_nn.optimize(lambda trial: objective_nn_men(trial, x_train_men_tr, y_train_men_tr, x_val_men, y_val_men),
                        n_trials=50)
best_params_men_nn = study_men_nn.best_params
print("Best NN Params (Men):", best_params_men_nn)

# Train final NN model for men with best hyperparameters
best_nn_men = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=tuple(best_params_men_nn[f"n_units_l{i}"] for i in range(best_params_men_nn["n_layers"])),
        activation=best_params_men_nn["activation"],
        alpha=best_params_men_nn["alpha"],
        learning_rate_init=best_params_men_nn["learning_rate_init"],
        max_iter=1000,
        random_state=42
    ))
])
best_nn_men.fit(x_train_men, y_train_men)
y_pred_men_nn = best_nn_men.predict_proba(x_test_men)[:, 1]
brier_men_nn = brier_score_loss(y_test_men, y_pred_men_nn)
print("Best NN Brier Score (Men):", brier_men_nn)


[I 2025-03-19 20:18:17,911] A new study created in memory with name: no-name-95df9c46-22d2-4a80-8562-d63f28ec2bcb
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:18:36,652] Trial 0 finished with value: 0.20076913948403527 and parameters: {'n_layers': 3, 'n_units_l0': 13, 'n_units_l1': 44, 'n_units_l2': 194, 'alpha': 0.0002057536484942702, 'learning_rate_init': 0.00021616668928489482, 'activation': 'relu'}. Best is trial 0 with value: 0.20076913948403527.
[I 2025-03-19 20:18:46,512] Trial 1 finished with value: 0.1814177452323555 and parameters: {'n_layers': 2, 'n_units_l0': 118, 'n_units_l1': 29, 'alpha': 0.0002665268175039287, 'learning_rate_init': 0.00031692998418523843, 'activation': 'tanh'}. Best is trial 1 with value: 0.1814177452323555.
[I 2025-03-19 20:18:47,693] Trial 2 finis

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:20:55,749] Trial 23 finished with value: 0.15282639898639538 and parameters: {'n_layers': 1, 'n_units_l0': 40, 'alpha': 0.0022551761186490886, 'learning_rate_init': 0.00010257470541475203, 'activation': 'relu'}. Best is trial 12 with value: 0.1502304332774823.
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:20:59,264] Trial 24 finished with value: 0.1711880331583375 and parameters: {'n_layers': 1, 'n_units_l0': 26, 'alpha': 0.013334576755701669, 'learning_rate_init': 0.00024070391205631344, 'activation': 'relu'}. Best is trial 12 with

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:22:52,750] Trial 43 finished with value: 0.1658773070452474 and parameters: {'n_layers': 1, 'n_units_l0': 21, 'alpha': 0.002918341863372483, 'learning_rate_init': 0.00046749712794218174, 'activation': 'tanh'}. Best is trial 28 with value: 0.13900642907167243.
C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:22:59,613] Trial 44 finished with value: 0.1538120301193064 and parameters: {'n_layers': 1, 'n_units_l0': 47, 'alpha': 0.0005049144223156866, 'learning_rate_init': 0.00014740106336196224, 'activation': 'tanh'}. Best is trial 28 with

Best NN Params (Women): {'n_layers': 1, 'n_units_l0': 87, 'alpha': 0.0009980421520108978, 'learning_rate_init': 0.0001015958297540628, 'activation': 'tanh'}


C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:23:40,545] A new study created in memory with name: no-name-19d67835-45fb-493c-877f-ba9b7eb27e80


Best NN Brier Score (Women): 0.18145764332289718


[I 2025-03-19 20:23:51,334] Trial 0 finished with value: 0.33918135567697705 and parameters: {'n_layers': 2, 'n_units_l0': 61, 'n_units_l1': 112, 'alpha': 0.013129555192475063, 'learning_rate_init': 0.0008255637630411846, 'activation': 'tanh'}. Best is trial 0 with value: 0.33918135567697705.
[I 2025-03-19 20:23:51,952] Trial 1 finished with value: 0.1984516593361832 and parameters: {'n_layers': 1, 'n_units_l0': 88, 'alpha': 0.04578841150680623, 'learning_rate_init': 0.09506508428261921, 'activation': 'relu'}. Best is trial 1 with value: 0.1984516593361832.
[I 2025-03-19 20:23:55,435] Trial 2 finished with value: 0.3191086796556407 and parameters: {'n_layers': 3, 'n_units_l0': 86, 'n_units_l1': 167, 'n_units_l2': 43, 'alpha': 0.010790284716862967, 'learning_rate_init': 0.023900849529669484, 'activation': 'tanh'}. Best is trial 1 with value: 0.1984516593361832.
[I 2025-03-19 20:24:03,474] Trial 3 finished with value: 0.3140342008610447 and parameters: {'n_layers': 2, 'n_units_l0': 125, 

C:\Users\Sebastian\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-03-19 20:26:21,484] Trial 29 finished with value: 0.26652874347137845 and parameters: {'n_layers': 2, 'n_units_l0': 148, 'n_units_l1': 198, 'alpha': 0.0017589027758019321, 'learning_rate_init': 0.0001076953428300234, 'activation': 'tanh'}. Best is trial 10 with value: 0.19020205404356896.
[I 2025-03-19 20:26:22,166] Trial 30 finished with value: 0.23745641091768407 and parameters: {'n_layers': 1, 'n_units_l0': 173, 'alpha': 0.009073738825744465, 'learning_rate_init': 0.06591235353387993, 'activation': 'relu'}. Best is trial 10 with value: 0.19020205404356896.
[I 2025-03-19 20:26:22,799] Trial 31 finished with value: 0.18539914518792233 and parameters: {'n_layers': 1, 'n_units_l0': 102, 'alpha': 0.07220485445993352, 'learning_rate_init': 0.063789

Best NN Params (Men): {'n_layers': 1, 'n_units_l0': 130, 'alpha': 0.027924330792765954, 'learning_rate_init': 0.04541527706385082, 'activation': 'relu'}
Best NN Brier Score (Men): 0.2914089860648776


## Ensamble

In [35]:
import numpy as np
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import brier_score_loss

# ------------------- For Women -------------------
# Define base estimators for women. They should already be tuned/trained.
estimators_women = [
    ('catboost', best_catboost_women),
    ('xgboost', best_xgb_women),
    ('logistic', best_lr_women),
    ('neural', best_nn_women),
]

# Build stacking classifier using a logistic regression meta-learner.
stacking_women = StackingClassifier(
    estimators=estimators_women,
    final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
    passthrough=False  # Change to True if you want meta-learner to see original features as well
)

# Define hyperparameter distribution for the meta-learner (final_estimator).
param_distributions_women = {
    'final_estimator__C': np.logspace(-4, 1, 50)
}

# Optimize meta-learner using RandomizedSearchCV
search_women = RandomizedSearchCV(
    estimator=stacking_women,
    param_distributions=param_distributions_women,
    n_iter=1,
    scoring='neg_log_loss',  # optimizing for well-calibrated probabilities
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_women.fit(x_train_women, y_train_women)
best_stacking_women = search_women.best_estimator_

# Evaluate on test set
y_pred_women_stacking = best_stacking_women.predict_proba(x_test_women)[:, 1]
brier_women_stacking = brier_score_loss(y_test_women, y_pred_women_stacking)
print("Best Stacking Classifier Brier Score (Women):", brier_women_stacking)


# ------------------- For Men -------------------
# Define base estimators for men. They should already be tuned/trained.
estimators_men = [
    ('catboost', best_catboost_men),
    ('xgboost', best_xgb_men),
    ('logistic', best_lr_men),
    ('neural', best_nn_men),
]

# Build stacking classifier using a logistic regression meta-learner.
stacking_men = StackingClassifier(
    estimators=estimators_men,
    final_estimator=LogisticRegression(solver='liblinear', max_iter=1000),
    passthrough=False
)

# Define hyperparameter distribution for the meta-learner.
param_distributions_men = {
    'final_estimator__C': np.logspace(-4, 1, 50)
}

# Optimize using RandomizedSearchCV
search_men = RandomizedSearchCV(
    estimator=stacking_men,
    param_distributions=param_distributions_men,
    n_iter=1,
    scoring='neg_log_loss',
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_men.fit(x_train_men, y_train_men)
best_stacking_men = search_men.best_estimator_

# Evaluate on test set
y_pred_men_stacking = best_stacking_men.predict_proba(x_test_men)[:, 1]
brier_men_stacking = brier_score_loss(y_test_men, y_pred_men_stacking)
print("Best Stacking Classifier Brier Score (Men):", brier_men_stacking)


Fitting 2 folds for each of 1 candidates, totalling 2 fits
0:	learn: 0.6797275	total: 11.5ms	remaining: 11.2s
1:	learn: 0.6633603	total: 21.2ms	remaining: 10.3s
2:	learn: 0.6523133	total: 30.9ms	remaining: 10s
3:	learn: 0.6426953	total: 40.3ms	remaining: 9.79s
4:	learn: 0.6305053	total: 50ms	remaining: 9.7s
5:	learn: 0.6216808	total: 59.7ms	remaining: 9.63s
6:	learn: 0.6143751	total: 69.3ms	remaining: 9.58s
7:	learn: 0.6050030	total: 78.6ms	remaining: 9.5s
8:	learn: 0.5965678	total: 88.3ms	remaining: 9.48s
9:	learn: 0.5883286	total: 98.2ms	remaining: 9.47s
10:	learn: 0.5796636	total: 108ms	remaining: 9.45s
11:	learn: 0.5705924	total: 118ms	remaining: 9.47s
12:	learn: 0.5633609	total: 128ms	remaining: 9.5s
13:	learn: 0.5557115	total: 138ms	remaining: 9.47s
14:	learn: 0.5472390	total: 148ms	remaining: 9.46s
15:	learn: 0.5409535	total: 158ms	remaining: 9.44s
16:	learn: 0.5356137	total: 167ms	remaining: 9.42s
17:	learn: 0.5288461	total: 177ms	remaining: 9.43s
18:	learn: 0.5212789	total: 18

174:	learn: 0.2306675	total: 1.81s	remaining: 8.26s
175:	learn: 0.2299577	total: 1.82s	remaining: 8.25s
176:	learn: 0.2292152	total: 1.83s	remaining: 8.25s
177:	learn: 0.2283352	total: 1.84s	remaining: 8.24s
178:	learn: 0.2275877	total: 1.85s	remaining: 8.22s
179:	learn: 0.2261235	total: 1.86s	remaining: 8.21s
180:	learn: 0.2248394	total: 1.87s	remaining: 8.2s
181:	learn: 0.2240710	total: 1.88s	remaining: 8.19s
182:	learn: 0.2235851	total: 1.89s	remaining: 8.18s
183:	learn: 0.2228401	total: 1.9s	remaining: 8.16s
184:	learn: 0.2223316	total: 1.91s	remaining: 8.15s
185:	learn: 0.2215731	total: 1.92s	remaining: 8.14s
186:	learn: 0.2210321	total: 1.93s	remaining: 8.12s
187:	learn: 0.2200873	total: 1.94s	remaining: 8.11s
188:	learn: 0.2184729	total: 1.95s	remaining: 8.1s
189:	learn: 0.2172901	total: 1.96s	remaining: 8.08s
190:	learn: 0.2165658	total: 1.97s	remaining: 8.07s
191:	learn: 0.2158822	total: 1.98s	remaining: 8.06s
192:	learn: 0.2150414	total: 1.99s	remaining: 8.05s
193:	learn: 0.2

346:	learn: 0.1165511	total: 3.61s	remaining: 6.54s
347:	learn: 0.1160525	total: 3.62s	remaining: 6.53s
348:	learn: 0.1156502	total: 3.63s	remaining: 6.52s
349:	learn: 0.1149271	total: 3.64s	remaining: 6.51s
350:	learn: 0.1143845	total: 3.65s	remaining: 6.5s
351:	learn: 0.1139326	total: 3.66s	remaining: 6.49s
352:	learn: 0.1135646	total: 3.67s	remaining: 6.47s
353:	learn: 0.1129798	total: 3.68s	remaining: 6.46s
354:	learn: 0.1124281	total: 3.69s	remaining: 6.45s
355:	learn: 0.1118491	total: 3.7s	remaining: 6.44s
356:	learn: 0.1114813	total: 3.71s	remaining: 6.43s
357:	learn: 0.1111398	total: 3.72s	remaining: 6.42s
358:	learn: 0.1105426	total: 3.73s	remaining: 6.41s
359:	learn: 0.1098027	total: 3.74s	remaining: 6.39s
360:	learn: 0.1094662	total: 3.75s	remaining: 6.38s
361:	learn: 0.1090780	total: 3.76s	remaining: 6.37s
362:	learn: 0.1083583	total: 3.77s	remaining: 6.36s
363:	learn: 0.1081084	total: 3.78s	remaining: 6.35s
364:	learn: 0.1074119	total: 3.79s	remaining: 6.34s
365:	learn: 0.

514:	learn: 0.0498639	total: 5.4s	remaining: 4.82s
515:	learn: 0.0496352	total: 5.41s	remaining: 4.81s
516:	learn: 0.0492978	total: 5.42s	remaining: 4.81s
517:	learn: 0.0489688	total: 5.44s	remaining: 4.8s
518:	learn: 0.0487964	total: 5.45s	remaining: 4.79s
519:	learn: 0.0485702	total: 5.46s	remaining: 4.78s
520:	learn: 0.0482753	total: 5.47s	remaining: 4.77s
521:	learn: 0.0478484	total: 5.49s	remaining: 4.76s
522:	learn: 0.0475308	total: 5.5s	remaining: 4.75s
523:	learn: 0.0472621	total: 5.51s	remaining: 4.74s
524:	learn: 0.0471386	total: 5.52s	remaining: 4.73s
525:	learn: 0.0469039	total: 5.53s	remaining: 4.72s
526:	learn: 0.0467237	total: 5.54s	remaining: 4.71s
527:	learn: 0.0465213	total: 5.55s	remaining: 4.7s
528:	learn: 0.0462904	total: 5.57s	remaining: 4.69s
529:	learn: 0.0460476	total: 5.58s	remaining: 4.68s
530:	learn: 0.0459449	total: 5.59s	remaining: 4.67s
531:	learn: 0.0457922	total: 5.6s	remaining: 4.66s
532:	learn: 0.0456512	total: 5.61s	remaining: 4.65s
533:	learn: 0.045

678:	learn: 0.0234651	total: 7.36s	remaining: 3.21s
679:	learn: 0.0233782	total: 7.37s	remaining: 3.2s
680:	learn: 0.0233182	total: 7.38s	remaining: 3.19s
681:	learn: 0.0232015	total: 7.39s	remaining: 3.18s
682:	learn: 0.0230747	total: 7.41s	remaining: 3.17s
683:	learn: 0.0229348	total: 7.41s	remaining: 3.15s
684:	learn: 0.0228379	total: 7.42s	remaining: 3.14s
685:	learn: 0.0227149	total: 7.43s	remaining: 3.13s
686:	learn: 0.0226140	total: 7.44s	remaining: 3.12s
687:	learn: 0.0225488	total: 7.45s	remaining: 3.11s
688:	learn: 0.0224139	total: 7.46s	remaining: 3.1s
689:	learn: 0.0223114	total: 7.47s	remaining: 3.09s
690:	learn: 0.0222273	total: 7.48s	remaining: 3.08s
691:	learn: 0.0221201	total: 7.5s	remaining: 3.06s
692:	learn: 0.0220363	total: 7.51s	remaining: 3.05s
693:	learn: 0.0219124	total: 7.52s	remaining: 3.04s
694:	learn: 0.0218017	total: 7.53s	remaining: 3.03s
695:	learn: 0.0217403	total: 7.54s	remaining: 3.02s
696:	learn: 0.0216559	total: 7.55s	remaining: 3.01s
697:	learn: 0.0

...

2075:	learn: 0.0760509	total: 10.5s	remaining: 3.66s
2076:	learn: 0.0759670	total: 10.5s	remaining: 3.65s
2077:	learn: 0.0759085	total: 10.6s	remaining: 3.65s
2078:	learn: 0.0758465	total: 10.6s	remaining: 3.64s
2079:	learn: 0.0757945	total: 10.6s	remaining: 3.64s
2080:	learn: 0.0757121	total: 10.6s	remaining: 3.63s
2081:	learn: 0.0756358	total: 10.6s	remaining: 3.63s
2082:	learn: 0.0755779	total: 10.6s	remaining: 3.62s
2083:	learn: 0.0755215	total: 10.6s	remaining: 3.62s
2084:	learn: 0.0754436	total: 10.6s	remaining: 3.61s
2085:	learn: 0.0753521	total: 10.6s	remaining: 3.61s
2086:	learn: 0.0752668	total: 10.6s	remaining: 3.61s
2087:	learn: 0.0751796	total: 10.6s	remaining: 3.6s
2088:	learn: 0.0751040	total: 10.6s	remaining: 3.6s
2089:	learn: 0.0750264	total: 10.6s	remaining: 3.59s
2090:	learn: 0.0749629	total: 10.6s	remaining: 3.59s
2091:	learn: 0.0749011	total: 10.7s	remaining: 3.58s
2092:	learn: 0.0748510	total: 10.7s	remaining: 3.58s
2093:	learn: 0.0747589	total: 10.7s	remaining: 3

2232:	learn: 0.0659376	total: 11.4s	remaining: 2.88s
2233:	learn: 0.0658711	total: 11.4s	remaining: 2.88s
2234:	learn: 0.0658075	total: 11.4s	remaining: 2.87s
2235:	learn: 0.0657508	total: 11.4s	remaining: 2.87s
2236:	learn: 0.0656913	total: 11.5s	remaining: 2.86s
2237:	learn: 0.0656317	total: 11.5s	remaining: 2.86s
2238:	learn: 0.0655904	total: 11.5s	remaining: 2.85s
2239:	learn: 0.0655375	total: 11.5s	remaining: 2.85s
2240:	learn: 0.0654637	total: 11.5s	remaining: 2.84s
2241:	learn: 0.0654136	total: 11.5s	remaining: 2.84s
2242:	learn: 0.0653597	total: 11.5s	remaining: 2.83s
2243:	learn: 0.0653227	total: 11.5s	remaining: 2.83s
2244:	learn: 0.0652533	total: 11.5s	remaining: 2.82s
2245:	learn: 0.0651996	total: 11.5s	remaining: 2.81s
2246:	learn: 0.0651434	total: 11.5s	remaining: 2.81s
2247:	learn: 0.0650792	total: 11.5s	remaining: 2.81s
2248:	learn: 0.0650267	total: 11.5s	remaining: 2.8s
2249:	learn: 0.0649699	total: 11.5s	remaining: 2.79s
2250:	learn: 0.0649024	total: 11.5s	remaining: 

2404:	learn: 0.0566652	total: 12.3s	remaining: 2.01s
2405:	learn: 0.0566226	total: 12.3s	remaining: 2s
2406:	learn: 0.0565499	total: 12.4s	remaining: 2s
2407:	learn: 0.0564980	total: 12.4s	remaining: 1.99s
2408:	learn: 0.0564640	total: 12.4s	remaining: 1.99s
2409:	learn: 0.0563899	total: 12.4s	remaining: 1.98s
2410:	learn: 0.0563440	total: 12.4s	remaining: 1.98s
2411:	learn: 0.0563005	total: 12.4s	remaining: 1.97s
2412:	learn: 0.0562741	total: 12.4s	remaining: 1.97s
2413:	learn: 0.0562460	total: 12.4s	remaining: 1.96s
2414:	learn: 0.0562168	total: 12.4s	remaining: 1.96s
2415:	learn: 0.0561654	total: 12.4s	remaining: 1.95s
2416:	learn: 0.0561284	total: 12.4s	remaining: 1.95s
2417:	learn: 0.0560925	total: 12.4s	remaining: 1.94s
2418:	learn: 0.0560324	total: 12.4s	remaining: 1.94s
2419:	learn: 0.0559863	total: 12.4s	remaining: 1.93s
2420:	learn: 0.0559486	total: 12.4s	remaining: 1.92s
2421:	learn: 0.0559168	total: 12.4s	remaining: 1.92s
2422:	learn: 0.0558740	total: 12.4s	remaining: 1.91s

2577:	learn: 0.0487878	total: 13.3s	remaining: 1.12s
2578:	learn: 0.0487571	total: 13.3s	remaining: 1.11s
2579:	learn: 0.0487137	total: 13.3s	remaining: 1.11s
2580:	learn: 0.0486834	total: 13.3s	remaining: 1.1s
2581:	learn: 0.0486486	total: 13.3s	remaining: 1.1s
2582:	learn: 0.0486114	total: 13.3s	remaining: 1.09s
2583:	learn: 0.0485488	total: 13.3s	remaining: 1.09s
2584:	learn: 0.0485020	total: 13.3s	remaining: 1.08s
2585:	learn: 0.0484646	total: 13.3s	remaining: 1.08s
2586:	learn: 0.0484188	total: 13.3s	remaining: 1.07s
2587:	learn: 0.0483741	total: 13.3s	remaining: 1.07s
2588:	learn: 0.0483272	total: 13.3s	remaining: 1.06s
2589:	learn: 0.0482668	total: 13.3s	remaining: 1.06s
2590:	learn: 0.0482326	total: 13.3s	remaining: 1.05s
2591:	learn: 0.0481861	total: 13.3s	remaining: 1.05s
2592:	learn: 0.0481510	total: 13.3s	remaining: 1.04s
2593:	learn: 0.0481169	total: 13.3s	remaining: 1.04s
2594:	learn: 0.0480655	total: 13.3s	remaining: 1.03s
2595:	learn: 0.0480008	total: 13.4s	remaining: 1

2746:	learn: 0.0425101	total: 14.2s	remaining: 253ms
2747:	learn: 0.0424844	total: 14.2s	remaining: 247ms
2748:	learn: 0.0424464	total: 14.2s	remaining: 242ms
2749:	learn: 0.0424201	total: 14.2s	remaining: 237ms
2750:	learn: 0.0423861	total: 14.2s	remaining: 232ms
2751:	learn: 0.0423450	total: 14.2s	remaining: 227ms
2752:	learn: 0.0423111	total: 14.2s	remaining: 222ms
2753:	learn: 0.0422693	total: 14.2s	remaining: 217ms
2754:	learn: 0.0422385	total: 14.2s	remaining: 211ms
2755:	learn: 0.0422143	total: 14.2s	remaining: 206ms
2756:	learn: 0.0421780	total: 14.2s	remaining: 201ms
2757:	learn: 0.0421509	total: 14.2s	remaining: 196ms
2758:	learn: 0.0421229	total: 14.2s	remaining: 191ms
2759:	learn: 0.0420934	total: 14.2s	remaining: 186ms
2760:	learn: 0.0420684	total: 14.2s	remaining: 180ms
2761:	learn: 0.0420416	total: 14.2s	remaining: 175ms
2762:	learn: 0.0419947	total: 14.2s	remaining: 170ms
2763:	learn: 0.0419583	total: 14.3s	remaining: 165ms
2764:	learn: 0.0419294	total: 14.3s	remaining: